# Constellation v2 — Model Story · Part 1: the Detector (Stage 1)

> **This is the v2 story and is deliberately separate from the v1 story**
> (`model/notebooks/model_story.ipynb`). v1 is a single end-to-end video
> classifier; v2 ("Constellation") is a 3-stage, landmark-primary pipeline.
> Don't merge the narratives — they're different models. This notebook covers
> **Stage 1** (the hand+head detector) and the decisions that got it validated.

## Why v2 exists

v1 was a tiny end-to-end video CNN trained on a narrow ASL Citizen subset. It
**overfit appearance** (signer, clothing, background, lighting) and plateaued at
low validation/test accuracy — it learned *who* and *where*, not *what the hands
do*.

**The v2 thesis:** recognize signs from **appearance-invariant pose geometry**
instead of pixels — hand keypoints + the hand's position relative to the head.
There's far less to overfit to. Getting geometry needs a pipeline, all **trained
from scratch, zero pretrained weights** (PRD Req 7):

```
Stage 1    detector        → hand & head BOXES         ← this notebook
Stage 1.5  landmark model  → 21 hand keypoints (crop)
Stage 2    recognizer      → sign, from pose geometry (+ small appearance stream)
```


## 1. Stage-1 design — a tiny single-shot detector

From-scratch, SSD-style single-shot detector, two foreground classes: `hand` (0)
and `head` (1).

- **Square anchors only** (BlazePalm trick): hands are roughly square blobs.
- **Depthwise-separable backbone**, stride-16, fully convolutional (adapts to any
  input size).
- **head ≈ face anchor:** the `head` class trains on **WIDER FACE** boxes — a fact
  that drives a key decision later.
- Data: **100DOH** (hand boxes) + **WIDER FACE** (head/face boxes), 90,204 train
  images, shrunk to 256 px for a ~1 GB Colab upload (lossless for the model's
  input resolution).


## 2. Iteration 1 — the first detector, and the test that "failed"

First model: **128² input, width 192** (~0.58 M params). It trained cleanly and
cleared the head proxy gate, but **missed the hand proxy gate** on 100DOH val
(`hand_dr 0.691` vs the 0.70 bar; `head_dr 0.883`).

Then the honest test — a **manually-labeled slice of 80 real ASL Citizen frames**
(hand + head boxes). The first read looked like a flat failure:

| class | ASL-audit detection-rate@0.5 | gate | first read |
|---|---|---|---|
| head | 0.287 | ≥ 0.80 | ❌ |
| hand | 0.514 | ≥ 0.60 | ❌ |

Instead of accepting or discarding the model, we **instrumented the failure**.


## 3. Diagnosis — what the numbers were really saying

Running predictions against the labels frame-by-frame told a very different story:

- **Head wasn't missing — it was mis-*measured*.** The detector found a head in
  **100 %** of frames, and its predicted center sat **inside** the labeled head
  box **90 %** of the time. It draws a tight **face** box (it learned from WIDER
  *face* data); we labeled whole **heads**. The face box sits inside the head box
  → IoU ≈ 0.46 → "miss" at the 0.5 threshold. The IoU metric was wrong for what
  the head box is *for*.
- **Hand was a real but specific weakness.** **0** frames had zero hand
  detections; per-hand recall was 0.68 @IoU0.5. The problem: a labeled hand
  averages ~150 px in a 640 px frame → only **~30 px** once squished to 128². Too
  few pixels to localize precisely.


## 4. Two decisions

**Decision A — score the head as a *normalization anchor*, not an object.**
Downstream (spec §3.6) the head box only provides a **center + scale** to
normalize hand positions. So the honest metric is *predicted center inside the GT
box + a sane size ratio* — not pixel-tight IoU. Whole-head labels stay; the
model's consistent face box is a fine anchor. (IoU@0.5 is kept as a *reference*
number, clearly flagged.)

**Decision B — give the hands more pixels.** Retrain at **img 128 → 192**
(≈2.25× pixels per hand) **and width 192 → 256**. To do this safely the anchor
scales were made to **auto-derive from the input size** (so they can never drift
out of sync), and the same resolution now flows through data → train → inference
→ the audit eval.

> Why resolution over more data: the detector already generalized (0 frame
> misses); the bottleneck was the *30-px hand*, which only more pixels fix.


## 5. Iteration 2 — the retrain (img 192, width 256)

![training curve](assets/detector_training_curve.png)

It converged at best_score **0.819** @ ep59
(`hand_dr 0.731`, `head_dr 0.906`) — and
crucially **hand_dr crossed the 0.70 proxy gate** that the first model missed.

| | iter 1 (128², w192) | **iter 2 (192², w256)** |
|---|---|---|
| params | 0.58 M | 1.02 M |
| 100DOH `hand_dr` | 0.691 ❌ | **0.731 ✅** |
| 100DOH `head_dr` | 0.883 | **0.906** |


## 6. Validated on real ASL frames

Re-running the ASL-audit gate on the retrained detector, with the **anchor metric**
for head (Decision A):

| metric (ASL-audit, binding gate) | value | gate | verdict |
|---|---|---|---|
| **head — anchor (center + scale)** | **0.838** | ≥ 0.80 | ✅ PASS |
| **hand — detection-rate@0.5** | **0.636** | ≥ 0.60 | ✅ PASS |
| head — IoU@0.5 (reference only) | 0.375 | — | face-box vs whole-head label |

**Both binding gates pass.** The detector places a correctly-centered, sanely-
sized head anchor and finds hands on real ASL frames — the higher-res retrain
lifted hand detection from 0.514 → 0.636.


In [ ]:
# Reproduce the validated gate from the saved eval
import json
a = json.load(open("../artifacts/checkpoints/detector/audit_eval.json"))
print(f"head anchor : {a['head_anchor_dr']:.3f}  (gate {a['head_gate']})  pass={a['head_pass']}")
print(f"hand dr     : {a['hand_dr']:.3f}  (gate {a['hand_gate']})  pass={a['hand_pass']}")
print(f"head IoU@0.5: {a['head_iou_dr']:.3f}  (reference; face box vs whole-head label)")


## 7. Live-webcam check (ADR-0001 live tester)

On the builder's own webcam (`scripts/detect_demo.py --mirror`) the detector
**correctly binds the head and both hands** in real time. Annotated audit frames
(blue = predicted face box + center · green = hand boxes · yellow = head→hand
vector in head-size units · red = the whole-head/hand labels):

![happy](assets/viz_07732970042675213-HAPPY.png)
![dog](assets/viz_06960268077531429-DOG.png)

The blue face box inside the red head label is the whole IoU "failure"; the yellow
head→hand vector — what Stage 2 consumes — is clean regardless.


## 8. Stage-1 takeaways (detector)

- **Instrument a "failure" before reacting** — the head fail was a metric/label
  mismatch, the hand fail a real resolution limit.
- **Match the metric to the box's job** — head box = normalization anchor, scored
  by center + scale, not IoU.
- **Pick the lever the diagnosis points at** — resolution (+ a little capacity)
  fixed the 30-px hand; both proxy and binding gates improved.

The full cross-stage decisions log is at the end. Next: the landmark model.


---
# Part 2 · Stage 1.5: the hand-landmark model

The detector gives a hand **box**; the recognizer needs hand **shape**. Stage 1.5
is a from-scratch **21-keypoint regressor** that runs inside each hand crop. This
is where the "lenient metric lied to us" lesson lives.


## 10. Design — a tiny per-crop keypoint regressor

- Input: the hand crop from the detector box, resized to **64×64** (a hand is
  small; 64 px is enough for joint geometry and keeps it fast).
- Output: **21 (x, y)** keypoints in crop coordinates (the standard hand topology),
  mapped back to frame coordinates for the geometry stage.
- Loss: **Wing loss** — designed for landmark regression (more gradient on the
  small/medium errors that matter for joints than plain L2).
- Data: **FreiHAND** (117 k single-hand images with 3-D-derived 2-D keypoints).
- From scratch, no pretrained weights — **MediaPipe is explicitly banned** (PRD).


## 11. Iteration 1 — the metric that lied

The first landmark model trained cleanly and reported **PCK@0.2 = 0.987** — a
near-perfect-looking score. But the live webcam test told the truth: the predicted
hand **collapsed toward a generic "mean hand"** — fingers roughly in the right
region, but not tracking the actual pose.

**Why the metric hid it:**
- **PCK@0.2 is lenient** — a keypoint counts as correct if it lands within 20 % of
  the hand size. A blurry mean-hand prediction clears that bar on most joints.
- **FreiHAND framing ≠ our framing.** FreiHAND hands are tightly, consistently
  cropped and centered; our detector boxes are looser and vary in scale/rotation.
  The model learned FreiHAND's *framing prior*, not pose from arbitrary crops.


## 12. Decision — a strict gate + framing-invariant training

**Decision C — judge it on PCK@0.1, and train it for our framing, not FreiHAND's.**

1. **Strict gate: PCK@0.1** (within 10 % of hand size) — a mean-hand can't fake
   this; joints have to actually be right.
2. **Keypoint-based square-window framing** to mimic the detector's real crops:
   train framing jitters scale **1.2–2.8×** the hand box and rotates **±25°**, so
   the model sees the loose, rotated, varied crops it gets at inference instead of
   FreiHAND's tidy ones.
3. **Width 64** crop input fixed across train/val/inference.


## 13. Iteration 2 — the retrain

![landmark curve](assets/landmark_training_curve.png)

Now the two PCK curves separate honestly: PCK@0.2 saturates near 0.97 early,
while the **strict PCK@0.1 climbs to 0.840** @ ep76
(mean error 0.061 of hand size). The gap between the lines is exactly the
collapse the first model hid behind.

| | iter 1 (FreiHAND framing) | **iter 2 (our framing + strict gate)** |
|---|---|---|
| reported metric | PCK@0.2 **0.987** (flattering) | PCK@0.1 **0.840** (honest) |
| real webcam hands | collapsed to mean hand ❌ | tracks finger pose ✅ |

On the audit frames the skeleton now follows the actual fingers:

![landmark audit](assets/lm_viz_001531801362371743-YELLOW.png)


In [ ]:
# Landmark: strict PCK@0.1 is the gate; PCK@0.2 kept only as a lenient reference
import json
h = json.load(open("../artifacts/checkpoints/landmark/history.json"))
b = h["history"][h["best_epoch"]]
print(f"best epoch    : {h['best_epoch']}")
print(f"PCK@0.1 (gate): {h['best_score']:.3f}")
print(f"PCK@0.2 (ref) : {b['pck_02']:.3f}   <- the number iter 1 hid behind")
print(f"mean err      : {b['mean_err']:.3f} (fraction of hand size)")


---
# Part 3 · Stage 2: the recognizer — and why geometry beats pixels

This is the payoff stage and the whole reason v2 exists. The recognizer classifies
a **sign** from the **pose geometry** the front-end produces — not from pixels.


## 14. Design — appearance-invariant geometry

Per frame we build a **93-dim vector** that is invariant to who is signing and
where they are:

- **84** = both hands × 21 keypoints × (x, y), expressed in a **head-centered,
  head-scaled** frame (the detector's head box is the normalization anchor).
- **+4** hand→head vectors, **+2** hand→hand, **+2** hand-presence, **+1**
  head-presence.

A clip is the sequence of these vectors. Because everything is normalized by the
head, the **signer's appearance, clothing, background, and camera distance fall
out** — there is far less to overfit to than raw pixels. That's the cure for v1's
disease.

Model **RecognizerA**: per-frame MLP → temporal head → sign logits, trained from
scratch on the ASL Citizen 75-sign subset with **signer-held-out** splits.


## 15. The climb — every gain measured, none from pretraining

![recog progression](assets/recog_progression.png)

| stage | val top-1 | test top-1 | what changed |
|---|---|---|---|
| **v1** (end-to-end pixels) | — | **18.0 %** | overfit appearance; the baseline to beat |
| v2 geometry baseline | 40.4 % | 37.6 % | pixels → head-normalized pose geometry |
| + aug + velocity + transformer | 45.7 % | 45.1 % | temporal modeling + thin-data regularization |
| **+ WLASL extra train** | **49.8%** | **47.9%** | more sign instances from a second corpus |

**18.0 % → 47.9% test, entirely from scratch.** At 47.9% top-1 the
test **top-3 ≈ 69 %** and **top-5 ≈ 78 %** — useful for a practice app that shows a
shortlist, while we keep pushing top-1.


## 16. Decisions — what moved the needle, and what didn't

**Decision D — geometry over pixels.** The single biggest jump (18 % → ~38–40 %)
came from *changing the input representation*, not the model. Normalizing by the
head box is what kills appearance overfitting.

**Decision E — "40 % is not acceptable" → push without pretraining.** When 40 %
fell short of production, the constraint held: **no pretrained models** (PRD Req 7).
So the gains had to come from *modeling and data*:
- **Augmentation** (temporal warp, pose rotation/scale, frame dropout) +
  **velocity features** (Δgeometry) + a **transformer temporal head** → +~7 pts.
- **More data** from a second corpus (**WLASL**), appended to **train only** (val/
  test stay pure ASL Citizen so the numbers stay comparable) → +~3 pts.

**Honest negative results.** Several architecture tweaks moved within the
**±5-pt single-run noise** on this small dataset — so every reported gain is a
**3-seed mean**, and tweaks inside the noise band were *not* claimed as wins.

**Decision F — the front-end is now the bottleneck.** With geometry+data tapped on
the current splits, the next lever is **feeding the recognizer cleaner geometry** —
better hand detection and landmark generalization on real ASL hands. That's what's
running now.


In [ ]:
# Recognizer: the validated from-scratch result (best-val seed)
import json
r = json.load(open("../artifacts/checkpoints/recog_a/history.json"))
print(f"best epoch : {r['best_epoch']}")
print(f"val  top-1 : {r['best_val_top1']:.3f}")
print(f"test top-1 : {r['test_top1']:.3f}   (vs v1's 0.180 end-to-end)")


---
# Part 4 · Plan 6 result — COCO-WholeBody front-end retrains (resolved: **not promoted**)

Decision F pointed at the front-end, so the detector and landmark were retrained
with **COCO-WholeBody** added (in-the-wild hands: boxes for the detector, 21
keypoints for the landmark). The hypothesis: messier real-world hands generalize to
the live app better than studio-clean ASL Citizen / tidy FreiHAND. COCO went to
**train only**; val/test stayed pure so the numbers stay comparable.

**Both retrains were evaluated honestly — and neither earned promotion.**

| stage | hypothesis | measured outcome | verdict |
|---|---|---|---|
| **Landmark** (+COCO) | better real-ASL keypoints | FreiHAND-val PCK@0.1 **0.840 → 0.826**; ASL audit frames **near-identical** (70-frame A/B) | ❌ wash — kept FreiHAND-only |
| **Detector** (+COCO) | higher hand recall | hand **0.636 → 0.671** ✅ **but** head-anchor **0.838 → 0.575** ❌ (below 0.80 gate) | ❌ disqualified — kept base |

**Why the detector broke (and why it's instructive).** The hand gain was *real* —
COCO hands work. But the merge added COCO images with **hand-only labels**, so the
**faces in those images became implicit negatives** → head confidence collapsed.
The head box is the **normalization anchor for all geometry**, so a head that's
wrong ~42 % of the time corrupts the recognizer's input — disqualifying regardless
of the hand gain. Both candidates are preserved (`*_coco/`) but the **validated
base detector + FreiHAND-only landmark stay in production.**

**The lesson:** cross-dataset transfer needs a **matching modality**. WLASL helped
(it's ASL signing video); COCO-WholeBody didn't (different domain). The fixable
path — label COCO **faces** as the `head` class too — is noted as Plan 6.1.


In [ ]:
# Reproduce the disqualifying detector regression (saved COCO candidate eval)
import json, os
p = "../artifacts/checkpoints/detector_coco/audit_eval.json"
if os.path.exists(p):
    c = json.load(open(p))
    b = json.load(open("../artifacts/checkpoints/detector/audit_eval.json"))
    print(f"hand dr      base {b['hand_dr']:.3f} -> COCO {c['hand_dr']:.3f}   (gain, real)")
    print(f"head anchor  base {b['head_anchor_dr']:.3f} -> COCO {c['head_anchor_dr']:.3f}   "
          f"(gate 0.80 -> {'PASS' if c['head_anchor_dr']>=0.8 else 'FAIL'})")
else:
    print("COCO candidate eval not present in this checkout")


---
# Part 5 · Pushing data further — MS-ASL (in progress)

With the front-end levers exhausted, the remaining ceiling is **data, not model
capacity**. The evidence is direct: **+WLASL gave +2.8 pts with zero model changes**,
while bigger models stayed inside the ±5-pt noise on this thin, signer-held-out
data — more parameters would overfit, not help.

So the next push is a second ASL corpus: **MS-ASL**.

- **74/75** of our signs are in MS-ASL — **~3,690 instances**, roughly **doubling**
  the training set (ASL Citizen 2,362 + WLASL 647).
- **Same modality as WLASL** (isolated ASL signing video) — the property COCO
  lacked, so transfer should actually work this time.
- **Train-only**, val/test stay pure ASL Citizen signer-held-out → the result stays
  directly comparable to **test 47.9% / top-3 ≈ 69 %**, the bar to beat.

The pipeline is built and running on Colab: a gloss-matched yt-dlp adapter
(`build_msasl_subset.py`) → cache geometry through the **same validated front-end** →
train RecognizerA (3 seeds). Expected realistically **+2–4 pts** after YouTube
link-rot, not a guarantee. **Result pending** — this section closes when the 3-seed
mean lands.


## Decisions log — the whole journey

1. **A "failing" gate isn't a failing model — instrument before reacting.**
   (Detector head fail = metric/label mismatch; hand fail = real resolution limit.)
2. **Match the metric to the box's job.** Head box is a normalization anchor →
   scored by center + scale, not IoU.
3. **Pick the lever the diagnosis points at.** Hands were resolution-limited → more
   pixels (img 192, width 256), verified by both proxy and binding gates.
4. **A lenient metric will lie to you.** Landmark PCK@0.2 = 0.987 hid a mean-hand
   collapse the live test exposed → switched to strict **PCK@0.1** + framing-
   invariant augmentation.
5. **Train for inference conditions, not the dataset's.** Landmark framing was
   jittered to match the detector's real, loose, rotated crops.
6. **Change the representation before the model.** Geometry over pixels is what
   broke v1's appearance overfitting (18 % → ~38 %).
7. **Constraints are non-negotiable; push within them.** "40 % isn't acceptable"
   *and* "no pretrained models" → gains came from augmentation, velocity, a
   transformer head, and a second corpus (WLASL).
8. **Report honest numbers.** ±5-pt single-run noise → 3-seed means; val/test kept
   pure when adding train-only corpora; tweaks inside the noise band not claimed.
9. **Kill experiments that don't earn promotion.** COCO-WholeBody retrains were run,
   measured, and **dropped** — the detector's hand gain didn't justify its head-anchor
   regression; the landmark was a wash. Validated front-end stays.
10. **Cross-dataset transfer needs a matching modality.** WLASL (ASL signing video)
    helped; COCO-WholeBody (different domain) didn't — same lesson, both directions.
11. **On thin data, scale data not parameters.** +WLASL gave a real +2.8 pts; bigger
    models stayed inside the noise. Next lever is MS-ASL (more data), not more params.
